# Integrations: MCP · LangChain · dbt

Drive Portiere from agent tools (MCP), agent frameworks (LangChain), and
analytics engineering (dbt) — all from the **one dependency-free tool core**
(`portiere.integrations.tools`). Every operation runs **offline**.

> **Prerequisites** — `pip install "portiere-health[polars]"`. The MCP and
> LangChain sections use extras (`[mcp]`, `[langchain]`) and degrade gracefully
> if not installed. Fully offline; deterministic; run top to bottom.

## 1. The shared tool core

Five offline operations. MCP and LangChain both wrap exactly these.

In [1]:
import json
from portiere.integrations.tools import PORTIERE_TOOLS, get_tool

for t in PORTIERE_TOOLS:
    req = t.input_schema.get('required', [])
    print(f"{t.name:34s} required={req}")

portiere_list_standards            required=[]
portiere_profile_source            required=['path']
portiere_suggest_schema_mapping    required=['columns']
portiere_map_concepts              required=['codes']
portiere_egress_posture            required=[]


### List target standards

In [2]:
print(get_tool('portiere_list_standards').run())

{'standards': ['fhir_r4', 'hl7v2_2.5.1', 'omop_cdm_v5.4', 'openehr_1.0.4']}


### Safety tool: verify no data can leave the machine

An agent can call this before acting to *prove* the no-egress posture.

In [3]:
posture = get_tool('portiere_egress_posture').run()
print(json.dumps(posture, indent=2))

{
  "offline": true,
  "components": [
    {
      "kind": "llm",
      "provider": "none",
      "verdict": "LOCAL"
    },
    {
      "kind": "embedding",
      "provider": "huggingface",
      "verdict": "LOCAL"
    },
    {
      "kind": "reranker",
      "provider": "huggingface",
      "verdict": "LOCAL"
    }
  ],
  "violations": []
}


### Suggest a schema mapping (offline, OMOP)

In [4]:
out = get_tool('portiere_suggest_schema_mapping').run(
    columns=[{'name': 'diagnosis_code'}, {'name': 'date_of_birth'}, {'name': 'sex'}],
    target_model='omop_cdm_v5.4',
)
for m in out['mappings']:
    print(f"  {m['source_column']:16s} -> {m['target_table']}.{m['target_column']}")

2026-07-25 08:22:31 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=none target_model=omop_cdm_v5.4


2026-07-25 08:22:32 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=none target_standard=omop_cdm_v5.4


2026-07-25 08:22:32 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=none


2026-07-25 08:22:32 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=137


2026-07-25 08:22:32 [info     ] local_schema_mapper.initialized standard=omop_cdm_v5.4


2026-07-25 08:22:32 [info     ] Stage 2 complete               auto=3 review=0 total=3


  diagnosis_code   -> condition_occurrence.condition_source_value
  date_of_birth    -> person.birth_datetime
  sex              -> person.gender_concept_id


### Map local codes to standard concepts (bundled demo vocabulary)

In [5]:
out = get_tool('portiere_map_concepts').run(
    codes=[{'code': 'E11.9', 'description': 'Type 2 diabetes'}],
    vocabularies=['ICD10CM'],
)
print(out.get('note', ''))
for it in out['items']:
    print(f"  {it['source_code']} -> {it['target_concept_id']} "
          f"{it['target_concept_name']} (conf={it['confidence']})")

2026-07-25 08:22:32 [info     ] athena.loading_concepts        path=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/src/portiere/_demo_data/vocabulary/CONCEPT.csv


2026-07-25 08:22:32 [info     ] athena.concepts_loaded         count=10 vocabularies=['ICD10CM']


2026-07-25 08:22:32 [info     ] athena.bm25s_built             count=10 path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_tool_gjxihw8v/index/concepts.json


2026-07-25 08:22:32 [info     ] athena.knowledge_layer_built   backend=bm25s concepts=10 paths={'bm25s_corpus_path': '/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_tool_gjxihw8v/index/concepts.json'}


2026-07-25 08:22:33 [info     ] PolarsEngine initialized      


...

2026-07-25 08:22:34 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=1


2026-07-25 08:22:34 [info     ] local_storage.concept_mapping_saved items_count=1 project=portiere-tool-concepts


2026-07-25 08:22:34 [info     ] project.concepts_mapped        auto_rate=100.0 project=portiere-tool-concepts total=1


2026-07-25 08:22:34 [warning  ] manifest.git_dirty             hint='Working tree has uncommitted changes; manifest records git_dirty=true' project_root=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/docs/notebooks_examples


no athena_dir/knowledge_path given — used the bundled demo vocabulary subset
  E11.9 -> 1567956 Type 2 diabetes mellitus without complications (conf=1.396)


## 2. MCP server

`portiere mcp` launches a stdio Model Context Protocol server exposing the
five tools. Below we show the registration without needing the `mcp` extra
installed, using the same `register_tools` the server uses.

In [6]:
from portiere.integrations.mcp import register_tools

class _Recorder:
    def __init__(self):
        self.tools = {}
    def tool(self, name=None, description=None):
        def deco(fn):
            self.tools[name] = description
            return fn
        return deco

rec = _Recorder()
register_tools(rec)
print('MCP tools that `portiere mcp` exposes:')
for name in rec.tools:
    print(' ', name)

# With the mcp extra installed, launch for real:
#   pip install 'portiere-health[mcp]'  &&  portiere mcp
# then point an MCP client (e.g. Claude Desktop) at it.

MCP tools that `portiere mcp` exposes:
  portiere_list_standards
  portiere_profile_source
  portiere_suggest_schema_mapping
  portiere_map_concepts
  portiere_egress_posture


## 3. LangChain tools

`get_langchain_tools()` returns the same operations as LangChain
`StructuredTool`s for use in agents/chains.

In [7]:
try:
    from portiere.integrations.langchain import get_langchain_tools
    tools = {t.name: t for t in get_langchain_tools()}
    print(f'{len(tools)} LangChain tools:', list(tools))
    print()
    print(tools['portiere_list_standards'].invoke({}))
    print(tools['portiere_egress_posture'].invoke({}))
except ImportError as exc:
    print('LangChain extra not installed — pip install "portiere-health[langchain]"')
    print(exc)

5 LangChain tools: ['portiere_list_standards', 'portiere_profile_source', 'portiere_suggest_schema_mapping', 'portiere_map_concepts', 'portiere_egress_posture']

{"standards": ["fhir_r4", "hl7v2_2.5.1", "omop_cdm_v5.4", "openehr_1.0.4"]}
{"offline": true, "components": [{"kind": "llm", "provider": "none", "verdict": "LOCAL"}, {"kind": "embedding", "provider": "huggingface", "verdict": "LOCAL"}, {"kind": "reranker", "provider": "huggingface", "verdict": "LOCAL"}], "violations": []}


## 4. dbt project generation

Turn a Portiere mapping into a runnable dbt project — one SQL model per
target table (select → rename → concept-lookup join), `schema.yml` tests,
and a `source_to_concept_map` seed. No dbt dependency to generate.

In [8]:
import tempfile
from pathlib import Path
from portiere.integrations.dbt import build_dbt_project
from portiere.models.schema_mapping import SchemaMapping, SchemaMappingItem
from portiere.models.concept_mapping import ConceptMapping, ConceptMappingItem

schema_map = SchemaMapping(items=[
    SchemaMappingItem(source_table='ehr_patient', source_column='patient_no',
                      target_table='person', target_column='person_id',
                      confidence=0.95, status='approved'),
    SchemaMappingItem(source_table='ehr_patient', source_column='sex',
                      target_table='person', target_column='gender_source_value',
                      confidence=0.9, status='approved'),
    SchemaMappingItem(source_table='ehr_dx', source_column='icd10',
                      target_table='condition_occurrence',
                      target_column='condition_source_value',
                      confidence=0.88, status='approved'),
])
concept_map = ConceptMapping(items=[
    ConceptMappingItem(source_code='M', source_column='sex', target_concept_id=8507,
                       target_concept_name='MALE', target_vocabulary_id='Gender',
                       confidence=0.99, method='auto'),
    ConceptMappingItem(source_code='F', source_column='sex', target_concept_id=8532,
                       target_concept_name='FEMALE', target_vocabulary_id='Gender',
                       confidence=0.99, method='auto'),
])

out = build_dbt_project(schema_map, concept_map,
                        Path(tempfile.mkdtemp()) / 'dbt_omop', source_schema='raw')
for f in sorted(out.rglob('*')):
    if f.is_file():
        print(' ', f.relative_to(out))

  README.md
  dbt_project.yml
  models/condition_occurrence.sql
  models/person.sql
  models/schema.yml
  seeds/source_to_concept_map.csv


### The generated `person.sql`

In [9]:
print((out / 'models' / 'person.sql').read_text())

-- Generated by Portiere — target: person (omop_cdm_v5.4)
-- Do not edit by hand; regenerate with `portiere dbt`.

with source as (
    select * from {{ source('raw', 'ehr_patient') }}
),

renamed as (
    select
        patient_no as person_id,
        sex as gender_source_value
    from source
),

mapped as (
    select
        renamed.*,
        coalesce(scm_0.target_concept_id, 0) as gender_source_value_concept_id
    from renamed
    left join {{ ref('source_to_concept_map') }} scm_0
        on scm_0.source_code = renamed.gender_source_value
       and scm_0.source_vocabulary_id = 'sex'
)

select * from mapped



### The `schema.yml` (tests on required OMOP fields)

In [10]:
print((out / 'models' / 'schema.yml').read_text())

version: 2
models:
- name: condition_occurrence
  description: Portiere-mapped condition_occurrence (omop_cdm_v5.4).
  columns:
  - name: condition_source_value
- name: person
  description: Portiere-mapped person (omop_cdm_v5.4).
  columns:
  - name: person_id
    tests:
    - not_null
  - name: gender_source_value
  - name: gender_source_value_concept_id



## 5. Domain-reranker decision rule

The v0.4.x decision harness (`scripts/reranker_spike.py`) applies a
pre-registered rule to candidate cross-encoders. Here on synthetic rows
(the real run needs the Athena export + model downloads):

In [11]:
import importlib.util
spec = importlib.util.spec_from_file_location(
    'reranker_spike', '../../scripts/reranker_spike.py')
rs = importlib.util.module_from_spec(spec); spec.loader.exec_module(rs)

incumbent = {'reranker_model': None, 'top_1': 0.303, 'mrr': 0.402}
candidates = [
    {'reranker_model': 'MedCPT', 'top_1': 0.320, 'mrr': 0.415},
    {'reranker_model': 'BioLORD', 'top_1': 0.305, 'mrr': 0.404},
]
print(rs.decide(incumbent, candidates)['reason'])

MedCPT beats incumbent: top-1 0.320 (+0.017), MRR 0.415 >= 0.402


## Recap

One offline tool core → MCP server (`portiere mcp`), LangChain tools
(`get_langchain_tools()`), and dbt generation (`portiere dbt`). Docs:
[MCP](../integrations/mcp.md) · [LangChain](../integrations/langchain.md) ·
[dbt](../integrations/dbt.md).